<a href="https://colab.research.google.com/github/Digital-AI-Finance/Cryptocurrency/blob/main/04_erc20_token_creation/notebooks/NB01_ERC20_Code_Demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lesson 4: ERC-20 Token Creation

This notebook covers ERC-20 tokens - the standard for fungible tokens on Ethereum:
- Understanding the ERC-20 standard
- Token functions (transfer, approve, transferFrom)
- Balance tracking
- Allowances and delegated transfers
- Interacting with tokens using web3.py

ERC-20 is the most widely used token standard - think USDT, USDC, DAI, and thousands more!

## Learning Objectives
By the end of this notebook, you will:
- Understand the ERC-20 token standard and its required functions
- Read and interpret ERC-20 contract code in Solidity
- Interact with tokens using Web3.py and ABIs
- Understand decimal handling for token amounts
- Master the transfer, approve, and transferFrom workflows
- Evaluate security considerations for token contracts

## Setup

Install web3.py for Ethereum interaction.

In [ ]:
!pip install -q web3

In [ ]:
from web3 import Web3
import json

print("All imports successful!")

## Part 1: Provider Configuration

Connect to Sepolia testnet using a public RPC endpoint.

In [ ]:
# Connect to Sepolia testnet via public RPC
PROVIDER_URL = "https://rpc.sepolia.org"
# Alternative: "https://sepolia.infura.io/v3/YOUR_PROJECT_ID"

w3 = Web3(Web3.HTTPProvider(PROVIDER_URL))

# Verify connection
if w3.is_connected():
    print("✓ Connected to Ethereum Sepolia testnet")
    print(f"  Chain ID: {w3.eth.chain_id}")
    print(f"  Latest block: {w3.eth.block_number:,}")
else:
    print("✗ Connection failed")

## Part 2: Understanding ERC-20 Standard

The ERC-20 standard defines a common interface that all tokens must implement.

**Required Functions:**
- `totalSupply()`: Total number of tokens
- `balanceOf(address)`: Get balance of an address
- `transfer(to, amount)`: Send tokens to another address
- `approve(spender, amount)`: Allow someone to spend your tokens
- `allowance(owner, spender)`: Check approved amount
- `transferFrom(from, to, amount)`: Transfer approved tokens

**Required Metadata:**
- `name()`: Token name (e.g., "MyToken")
- `symbol()`: Token symbol (e.g., "MTK")
- `decimals()`: Number of decimal places (usually 18)

## Part 3: ERC-20 Contract Code

Here's the Solidity code for our MyToken contract:

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.0;

contract MyToken {
    string public name = "MyToken";
    string public symbol = "MTK";
    uint8 public decimals = 18;
    uint256 public totalSupply;
    
    mapping(address => uint256) public balanceOf;
    mapping(address => mapping(address => uint256)) public allowance;
    
    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);
    
    constructor(uint256 _initialSupply) {
        totalSupply = _initialSupply * 10**decimals;
        balanceOf[msg.sender] = totalSupply;
    }
    
    function transfer(address _to, uint256 _value) public returns (bool) {
        require(balanceOf[msg.sender] >= _value, "Insufficient balance");
        balanceOf[msg.sender] -= _value;
        balanceOf[_to] += _value;
        emit Transfer(msg.sender, _to, _value);
        return true;
    }
    
    function approve(address _spender, uint256 _value) public returns (bool) {
        allowance[msg.sender][_spender] = _value;
        emit Approval(msg.sender, _spender, _value);
        return true;
    }
    
    function transferFrom(address _from, address _to, uint256 _value) public returns (bool) {
        require(balanceOf[_from] >= _value, "Insufficient balance");
        require(allowance[_from][msg.sender] >= _value, "Insufficient allowance");
        balanceOf[_from] -= _value;
        balanceOf[_to] += _value;
        allowance[_from][msg.sender] -= _value;
        emit Transfer(_from, _to, _value);
        return true;
    }
}
```

## Part 4: ERC-20 Token ABI

The ABI (Application Binary Interface) defines how to interact with the token contract.

In [ ]:
# Standard ERC-20 ABI
ERC20_ABI = json.loads('''[
    {"inputs":[],"name":"name","outputs":[{"type":"string"}],"stateMutability":"view","type":"function"},
    {"inputs":[],"name":"symbol","outputs":[{"type":"string"}],"stateMutability":"view","type":"function"},
    {"inputs":[],"name":"decimals","outputs":[{"type":"uint8"}],"stateMutability":"view","type":"function"},
    {"inputs":[],"name":"totalSupply","outputs":[{"type":"uint256"}],"stateMutability":"view","type":"function"},
    {"inputs":[{"name":"account","type":"address"}],"name":"balanceOf","outputs":[{"type":"uint256"}],"stateMutability":"view","type":"function"},
    {"inputs":[{"name":"to","type":"address"},{"name":"amount","type":"uint256"}],"name":"transfer","outputs":[{"type":"bool"}],"stateMutability":"nonpayable","type":"function"},
    {"inputs":[{"name":"spender","type":"address"},{"name":"amount","type":"uint256"}],"name":"approve","outputs":[{"type":"bool"}],"stateMutability":"nonpayable","type":"function"},
    {"inputs":[{"name":"owner","type":"address"},{"name":"spender","type":"address"}],"name":"allowance","outputs":[{"type":"uint256"}],"stateMutability":"view","type":"function"},
    {"inputs":[{"name":"from","type":"address"},{"name":"to","type":"address"},{"name":"amount","type":"uint256"}],"name":"transferFrom","outputs":[{"type":"bool"}],"stateMutability":"nonpayable","type":"function"},
    {"anonymous":false,"inputs":[{"indexed":true,"name":"from","type":"address"},{"indexed":true,"name":"to","type":"address"},{"indexed":false,"name":"value","type":"uint256"}],"name":"Transfer","type":"event"},
    {"anonymous":false,"inputs":[{"indexed":true,"name":"owner","type":"address"},{"indexed":true,"name":"spender","type":"address"},{"indexed":false,"name":"value","type":"uint256"}],"name":"Approval","type":"event"}
]''')

print("ERC-20 ABI loaded!")
print("\nThis ABI allows us to interact with ANY ERC-20 token.")

## Part 5: Interacting with a Real ERC-20 Token

Let's interact with a well-known ERC-20 token on the testnet.

**Note**: For demonstration, we'll show the interaction pattern. You would need:
1. An actual token contract address on Sepolia
2. A wallet with some tokens
3. Testnet ETH for gas fees

In [ ]:
# Example: How to interact with an ERC-20 token
# (You would replace this with an actual deployed token address)

print("=" * 70)
print("ERC-20 TOKEN INTERACTION PATTERN")
print("=" * 70)
print("""
To interact with an ERC-20 token:

1. CREATE CONTRACT INSTANCE:
   token = w3.eth.contract(address=TOKEN_ADDRESS, abi=ERC20_ABI)

2. READ TOKEN METADATA (free):
   name = token.functions.name().call()
   symbol = token.functions.symbol().call()
   decimals = token.functions.decimals().call()
   total_supply = token.functions.totalSupply().call()

3. CHECK BALANCE (free):
   balance = token.functions.balanceOf(YOUR_ADDRESS).call()
   human_readable = balance / 10**decimals

4. TRANSFER TOKENS (costs gas):
   amount = 100 * 10**decimals  # 100 tokens
   tx_hash = token.functions.transfer(RECIPIENT_ADDRESS, amount).transact({
       'from': YOUR_ADDRESS
   })

5. APPROVE SPENDER (costs gas):
   tx_hash = token.functions.approve(SPENDER_ADDRESS, amount).transact({
       'from': YOUR_ADDRESS
   })

6. CHECK ALLOWANCE (free):
   allowed = token.functions.allowance(OWNER_ADDRESS, SPENDER_ADDRESS).call()

7. TRANSFER FROM (costs gas, requires approval):
   tx_hash = token.functions.transferFrom(FROM_ADDRESS, TO_ADDRESS, amount).transact({
       'from': SPENDER_ADDRESS
   })
""")
print("=" * 70)

## Part 6: Understanding Decimals

Most ERC-20 tokens use 18 decimals (like ETH). This means:
- 1 token = 1,000,000,000,000,000,000 wei (10^18)
- Internally stored as integers (no floating point)
- Converted to human-readable format for display

In [ ]:
# Decimal conversion example
decimals = 18

# Human-readable to contract format
human_amount = 100.5  # 100.5 tokens
contract_amount = int(human_amount * 10**decimals)

print("=" * 70)
print("TOKEN DECIMAL CONVERSION")
print("=" * 70)
print(f"\nHuman-readable: {human_amount} tokens")
print(f"Contract format: {contract_amount} wei")
print(f"Scientific notation: {contract_amount:.2e}")

# Contract format to human-readable
back_to_human = contract_amount / 10**decimals
print(f"\nConverted back: {back_to_human} tokens")

print("\nWhy 18 decimals?")
print("  - Matches ETH's precision")
print("  - Allows fractional transfers")
print("  - No floating-point arithmetic (uses integers)")
print("=" * 70)

## Part 7: Transfer Workflow

Let's visualize how token transfers work.

In [ ]:
print("=" * 70)
print("TOKEN TRANSFER WORKFLOW")
print("=" * 70)
print("""
SCENARIO: Alice wants to send 100 MTK tokens to Bob

INITIAL STATE:
  Alice balance: 1000 MTK
  Bob balance:     50 MTK

STEP 1 - Alice calls transfer():
  token.functions.transfer(BOB_ADDRESS, 100 * 10**18).transact({'from': ALICE})

STEP 2 - Contract checks:
  ✓ Does Alice have >= 100 MTK? YES (she has 1000)
  ✓ Is the recipient address valid? YES

STEP 3 - Contract updates balances:
  balanceOf[Alice] -= 100 MTK  →  900 MTK
  balanceOf[Bob]   += 100 MTK  →  150 MTK

STEP 4 - Contract emits event:
  emit Transfer(Alice, Bob, 100 MTK)

FINAL STATE:
  Alice balance: 900 MTK
  Bob balance:   150 MTK

GAS COST: ~50,000 gas (~$1-5 depending on gas price)
""")
print("=" * 70)

## Part 8: Approve/TransferFrom Workflow

The approve/transferFrom pattern enables delegated transfers.

In [ ]:
print("=" * 70)
print("APPROVE/TRANSFERFROM WORKFLOW")
print("=" * 70)
print("""
SCENARIO: Alice wants to allow a DEX contract to spend 500 MTK

WHY USE THIS?
  - Smart contracts can't pull tokens without approval
  - Used by DEXs, lending protocols, etc.
  - Enables complex automated interactions

STEP 1 - Alice approves the DEX:
  token.functions.approve(DEX_ADDRESS, 500 * 10**18).transact({'from': ALICE})
  
  Result: allowance[Alice][DEX] = 500 MTK

STEP 2 - DEX calls transferFrom():
  token.functions.transferFrom(ALICE, BOB, 200 * 10**18).transact({'from': DEX})

STEP 3 - Contract checks:
  ✓ Does Alice have >= 200 MTK? YES
  ✓ Has Alice approved DEX for >= 200 MTK? YES (approved 500)

STEP 4 - Contract updates:
  balanceOf[Alice] -= 200 MTK
  balanceOf[Bob]   += 200 MTK
  allowance[Alice][DEX] -= 200 MTK  →  300 MTK remaining

SECURITY NOTE:
  ⚠️  Only approve trusted contracts!
  ⚠️  Approved contracts can spend up to the allowance
  ⚠️  Revoke approval by setting to 0
""")
print("=" * 70)

### TODO Exercise 1

Practice decimal conversions:
1. Convert 50.25 tokens to wei (18 decimals)
2. Convert 1500000000000000000 wei back to tokens
3. What's the smallest transferable amount with 18 decimals?

In [ ]:
# TODO: Practice decimal conversions
# decimals = 18
# amount_tokens = 50.25
# amount_wei = int(amount_tokens * 10**decimals)
# print(f"{amount_tokens} tokens = {amount_wei} wei")

## Part 9: Common Token Operations Summary

In [ ]:
print("=" * 70)
print("COMMON ERC-20 TOKEN OPERATIONS")
print("=" * 70)
print("""
1. GET TOKEN INFO (Free):
   name = token.functions.name().call()
   symbol = token.functions.symbol().call()
   decimals = token.functions.decimals().call()

2. CHECK BALANCE (Free):
   balance = token.functions.balanceOf(address).call()

3. SEND TOKENS (Costs Gas):
   token.functions.transfer(recipient, amount).transact({'from': sender})

4. APPROVE SPENDER (Costs Gas):
   token.functions.approve(spender, amount).transact({'from': owner})

5. CHECK ALLOWANCE (Free):
   allowed = token.functions.allowance(owner, spender).call()

6. DELEGATED TRANSFER (Costs Gas):
   token.functions.transferFrom(from, to, amount).transact({'from': spender})

COMMON USE CASES:
  - Payments: Simple transfer()
  - DEX trading: approve() + transferFrom()
  - Staking: approve() + contract deposits tokens
  - Lending: approve() + contract manages collateral
""")
print("=" * 70)

## Part 10: Security Considerations

In [ ]:
print("=" * 70)
print("ERC-20 SECURITY BEST PRACTICES")
print("=" * 70)
print("""
FOR USERS:
  ✓ Only approve trusted contracts
  ✓ Check token contract is verified on Etherscan
  ✓ Revoke unnecessary approvals
  ✓ Use tools like revoke.cash to manage approvals
  ⚠️  Unlimited approvals = unlimited risk

FOR DEVELOPERS:
  ✓ Use SafeMath or Solidity 0.8+ (overflow protection)
  ✓ Follow Checks-Effects-Interactions pattern
  ✓ Emit events for all state changes
  ✓ Test edge cases (zero amounts, self-transfers)
  ✓ Consider ERC-20 approval race condition

COMMON VULNERABILITIES:
  • Approval race condition (approve → front-run → approve)
  • Integer overflow/underflow (pre-0.8.0 Solidity)
  • Reentrancy in token callbacks
  • Missing return value checks

AUDITING:
  - Get professional audits before mainnet
  - Use battle-tested libraries (OpenZeppelin)
  - Test on testnets extensively
""")
print("=" * 70)

### TODO Exercise 2

Research real ERC-20 tokens:
1. Look up USDC or DAI on Etherscan
2. Find their contract addresses
3. Check total supply and holder count
4. View recent transfers

## Part 11: Real-World ERC-20 Examples

In [ ]:
print("=" * 70)
print("FAMOUS ERC-20 TOKENS")
print("=" * 70)
print("""
STABLECOINS:
  • USDT (Tether) - Market cap: $90B+
  • USDC (USD Coin) - Market cap: $25B+
  • DAI (MakerDAO) - Market cap: $5B+
    → Use case: Stable value for payments, trading

DEFI TOKENS:
  • UNI (Uniswap) - DEX governance token
  • AAVE (Aave) - Lending protocol token
  • LINK (Chainlink) - Oracle network token
    → Use case: Governance, staking, utility

EXCHANGE TOKENS:
  • BNB (Binance) - Exchange utility token
  • CRO (Crypto.com) - Exchange rewards token
    → Use case: Fee discounts, rewards

WRAPPED TOKENS:
  • WBTC (Wrapped Bitcoin) - Bitcoin on Ethereum
  • WETH (Wrapped ETH) - ETH as ERC-20
    → Use case: Use non-ERC-20 assets in DeFi

WHY SO POPULAR?
  ✓ Standardized interface
  ✓ Works with all wallets and exchanges
  ✓ Easy to integrate into DeFi protocols
  ✓ Composable with other contracts
""")
print("=" * 70)

## Key Takeaways

1. **ERC-20** is the standard for fungible tokens on Ethereum
2. **Core functions**: transfer, approve, transferFrom
3. **Decimals** (usually 18) enable fractional amounts
4. **Approve pattern** enables delegated transfers for DeFi
5. **Events** (Transfer, Approval) log all state changes
6. **Security** is critical - only approve trusted contracts
7. **Standardization** enables interoperability across ecosystem
8. **Real-world usage** includes stablecoins, DeFi, governance

## Next Steps

To deploy your own token:
1. Use OpenZeppelin's ERC-20 implementation
2. Test on a testnet first
3. Get a security audit before mainnet
4. Consider token economics carefully
5. Implement proper access controls

Congratulations! You now understand the foundation of the token economy!